## AlexNet: Building and Training the 2012 ImageNet Winner

This notebook walks through AlexNet (Krizhevsky, Sutskever, Hinton, 2012), the
convolutional network whose ILSVRC-2012 win is widely credited with starting the
modern deep learning era in computer vision.

We cover, in order:

1. The basic idea, and why it mattered
2. Exploring the dataset (CIFAR-10 stands in for ImageNet)
3. The architecture, layer by layer
4. Building AlexNet in PyTorch
5. Training it on a small subset of data

This continues from [need_for_cnn.ipynb](need_for_cnn.ipynb): we already saw
convolution beat a plain DNN on images. AlexNet is the architecture that proved just
how far that idea could scale.

### 1. The basic idea

Before 2012, the best ImageNet classifiers used hand-engineered features (SIFT, HOG,
Fisher vectors) feeding into an SVM. AlexNet cut the ILSVRC-2012 top-5 error from
about 26% to 15.3%, trained end to end directly on raw pixels, and is usually
credited with turning the field toward deep learning for vision.

Convolutional networks were not new (LeNet-5, 1998, is the same lineage as
[need_for_cnn.ipynb](need_for_cnn.ipynb)'s small network). What made AlexNet work at
ImageNet scale was a combination of practical ideas, several still standard today:

| Idea | What it did |
| --- | --- |
| ReLU activation | Replaced tanh/sigmoid; doesn't saturate for positive inputs, so training is much faster. |
| Trained across 2 GPUs | Split across two 3GB GTX 580s because it didn't fit on one — hence the paper's "communicating"/"independent" layer halves. A modern GPU, or our CPU demo, holds it easily. |
| Overlapping max pooling | Pooling stride (2) smaller than window size (3); a small but measurable accuracy gain over non-overlapping pooling. |
| Local Response Normalization (LRN) | Normalizes each unit against nearby feature maps. Later replaced by Batch Normalization; kept here for historical accuracy. |
| Dropout | Zeroes 50% of units in the two largest FC layers during training. Needed because the classifier head holds most of the network's parameters (section 3). |
| Data augmentation | Random crops, flips, and PCA color jitter to grow the effective training set. |
| Depth | 5 conv + 3 FC = 8 learned layers — deep for its time. |

Training the full setup (1.2M images, 1000 classes) isn't practical in a classroom.
Section 5 trains the same architecture on a tiny slice of a much smaller dataset, to
make the overfitting risk above concrete.

### 2. Explore the dataset

AlexNet was built for ImageNet (1.2M training images, 1000 classes, 224x224x3). We
use `CIFAR-10` instead: 60,000 32x32 color photos across 10 classes (airplane,
automobile, bird, cat, deer, dog, frog, horse, ship, truck), 50,000 train / 10,000
test.

CIFAR-10 is much smaller and lower-resolution, but keeps what matters here: real
color photographs with visually similar classes. We resize the 32x32 images up to
224x224 so the original architecture runs unmodified — this adds no real detail, but
lets us use the exact paper architecture.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cpu")
print("device:", device)

CIFAR-10's official mirror (`cs.toronto.edu`) is a single ~170MB archive and known
to be slow. As in [need_for_cnn.ipynb](need_for_cnn.ipynb)'s FashionMNIST download,
we instead fetch pre-sorted JPEGs (`train/<class>/`, `test/<class>/`) from an
actively maintained GitHub mirror
([YoongiKim/CIFAR-10-images](https://github.com/YoongiKim/CIFAR-10-images)), which
also lets us load it with torchvision's generic `ImageFolder` instead of
CIFAR-10-specific code.

In [ ]:
import os

CIFAR_URL = "https://codeload.github.com/YoongiKim/CIFAR-10-images/tar.gz/refs/heads/master"
DATA_DIR = "data/cifar10"

if os.path.isdir(f"{DATA_DIR}/train"):
    print(f"{DATA_DIR} already present, skipping download")
else:
    !mkdir -p {DATA_DIR}
    !curl -sS -L -o /tmp/cifar10_images.tar.gz {CIFAR_URL}
    !tar -xzf /tmp/cifar10_images.tar.gz -C {DATA_DIR} --strip-components=1
    !rm /tmp/cifar10_images.tar.gz

!ls {DATA_DIR}

In [ ]:
train_dir = f"{DATA_DIR}/train"
test_dir = f"{DATA_DIR}/test"

# no transform yet: raw PIL images, for visualization only
raw_train_ds = datasets.ImageFolder(root=train_dir)
raw_test_ds = datasets.ImageFolder(root=test_dir)
class_names = raw_train_ds.classes

print(f"train images: {len(raw_train_ds)}, test images: {len(raw_test_ds)}")
print(f"classes: {class_names}")
print(f"one image: {raw_train_ds[0][0].size}, mode: {raw_train_ds[0][0].mode}")

In [ ]:
# ImageFolder lists images in class-sorted order, so we sample random indices here
# rather than taking the first 16 (which would all be the same class)
sample_idx = np.random.RandomState(0).choice(len(raw_train_ds), size=16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, idx in zip(axes.flat, sample_idx):
    img, label = raw_train_ds[idx]
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 images (32x32 native resolution)")
plt.tight_layout()
plt.show()

In [ ]:
counts = np.bincount(raw_train_ds.targets)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_names, counts)
ax.set_ylabel("training images")
ax.set_title("CIFAR-10 is class-balanced: 5,000 training images per class")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3. The architecture, layer by layer

The table below assumes a 224x224x3 input (conventionally cropped from 256x256; we
resize CIFAR-10's 32x32 images up to 224x224 in section 5 instead). Rather than quote
numbers from the paper, we trace output shape and parameter count through a real
forward pass.

In [ ]:
feature_layers = [
    ("conv1", nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2)),
    ("relu1", nn.ReLU(inplace=True)),
    ("lrn1", nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2)),
    ("pool1", nn.MaxPool2d(kernel_size=3, stride=2)),
    ("conv2", nn.Conv2d(96, 256, kernel_size=5, padding=2)),
    ("relu2", nn.ReLU(inplace=True)),
    ("lrn2", nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2)),
    ("pool2", nn.MaxPool2d(kernel_size=3, stride=2)),
    ("conv3", nn.Conv2d(256, 384, kernel_size=3, padding=1)),
    ("relu3", nn.ReLU(inplace=True)),
    ("conv4", nn.Conv2d(384, 384, kernel_size=3, padding=1)),
    ("relu4", nn.ReLU(inplace=True)),
    ("conv5", nn.Conv2d(384, 256, kernel_size=3, padding=1)),
    ("relu5", nn.ReLU(inplace=True)),
    ("pool3", nn.MaxPool2d(kernel_size=3, stride=2)),
]

def count_params(module):
    return sum(p.numel() for p in module.parameters())

x = torch.zeros(1, 3, 224, 224)
print(f"{'layer':<8}{'output shape':<20}{'params':>12}")
print(f"{'input':<8}{str(tuple(x.shape)):<20}{'':>12}")

conv_total = 0
for name, layer in feature_layers:
    x = layer(x)
    p = count_params(layer)
    conv_total += p
    print(f"{name:<8}{str(tuple(x.shape)):<20}{p:>12,}")

print(f"\nfeature map entering the classifier: {tuple(x.shape)}"
      f" -> flattens to {x.numel():,} values")
print(f"total convolutional parameters: {conv_total:,}")

The last pooling layer leaves a 6x6x256 feature map — exactly the `256 * 6 * 6 =
9216` input the paper's classifier expects. Now trace the three fully connected
layers.

In [ ]:
classifier_layers = [
    ("fc6", nn.Linear(256 * 6 * 6, 4096)),
    ("fc7", nn.Linear(4096, 4096)),
    ("fc8", nn.Linear(4096, 1000)),  # 1000 ImageNet classes; we'll use 10 for CIFAR-10 below
]

xf = x.flatten(1)
print(f"{'layer':<8}{'output shape':<20}{'params':>14}")
print(f"{'flatten':<8}{str(tuple(xf.shape)):<20}{'':>14}")

fc_total = 0
for name, layer in classifier_layers:
    xf = layer(xf)
    p = count_params(layer)
    fc_total += p
    print(f"{name:<8}{str(tuple(xf.shape)):<20}{p:>14,}")

print(f"\ntotal fully-connected parameters (1000-class head): {fc_total:,}")
print(f"grand total: {conv_total + fc_total:,}")
print(f"\nfc6 + fc7 alone account for "
      f"{100 * (classifier_layers[0][1].weight.numel() + classifier_layers[1][1].weight.numel()) / (conv_total + fc_total):.1f}%"
      f" of all parameters")

Two things stand out from these numbers:

- The five convolutional layers together hold only a few million parameters, and
  their cost does not scale with the classifier's width, only with kernel size and
  channel count, exactly the property explored for a small conv layer in
  [need_for_cnn.ipynb](need_for_cnn.ipynb).
- `fc6` and `fc7` alone hold the large majority of the network's parameters. This is
  exactly why the paper leans so heavily on Dropout: the overfitting risk is
  concentrated almost entirely in the classifier head, not the feature extractor.

### 4. Building AlexNet in PyTorch

We package the layer sequence above into a reusable `nn.Module`, with two changes
from the raw trace in section 3:

- `num_classes` is a constructor argument (10 for CIFAR-10, not 1000).
- `nn.AdaptiveAvgPool2d((6, 6))` sits right before the classifier. With a fixed
  224x224 input this changes nothing (the last conv block already outputs 6x6), but
  it makes the model robust to other input sizes — the same trick
  `torchvision.models.alexnet` uses.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2), nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


alexnet = AlexNet(num_classes=10).to(device)
print(alexnet)
print(f"\nAlexNet parameters (10-class head): {count_params(alexnet):,}")

In [ ]:
# sanity check: a batch of 224x224 RGB images in, 10 class scores out
dummy = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    out = alexnet(dummy)
print("input shape:", tuple(dummy.shape), "-> output shape:", tuple(out.shape))

### 5. Training on a small subset

Training on the full 50,000-image CIFAR-10 training set would take a while on CPU.
To keep this runnable in class, we sample a small, class-balanced subset instead: a
fixed number of images per class for train and test.

This is intentionally too little data for a model with tens of millions of
parameters — expect visible overfitting, the exact failure mode `Dropout` and data
augmentation exist to fight (see
[overfit_underfit_regularization.ipynb](overfit_underfit_regularization.ipynb)). The
goal isn't a high accuracy number, it's watching the real architecture train end to
end and making section 3's parameter-count risk concrete.

In [ ]:
N_TRAIN_PER_CLASS = 30   # 300 training images total
N_TEST_PER_CLASS = 20    # 200 test images total
BATCH_SIZE = 32
EPOCHS = 25


In [ ]:
# CIFAR-10 per-channel mean/std, computed over the full training set
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds_full = datasets.ImageFolder(root=train_dir, transform=transform)
test_ds_full = datasets.ImageFolder(root=test_dir, transform=transform)


In [ ]:
# ImageFolder has no built-in "N per class" split, so sample our own balanced
# subset of indices instead of using the (much larger) full datasets directly.
def balanced_subset_indices(dataset, per_class, num_classes=10, seed=0):
    targets = np.array(dataset.targets)
    rng = np.random.RandomState(seed)
    indices = []
    for c in range(num_classes):
        cls_idx = np.where(targets == c)[0]
        rng.shuffle(cls_idx)
        indices.extend(cls_idx[:per_class].tolist())
    rng.shuffle(indices)
    return indices


train_idx = balanced_subset_indices(train_ds_full, N_TRAIN_PER_CLASS)
test_idx = balanced_subset_indices(test_ds_full, N_TEST_PER_CLASS)

train_ds = Subset(train_ds_full, train_idx)
test_ds = Subset(test_ds_full, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train subset: {len(train_ds)} images ({N_TRAIN_PER_CLASS}/class)")
print(f"test subset:  {len(test_ds)} images ({N_TEST_PER_CLASS}/class)")


In [ ]:
def train_model(model, loader, epochs, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"loss": [], "train_acc": [], "test_acc": []}
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)

        epoch_loss, epoch_acc = running_loss / total, correct / total
        epoch_test_acc = evaluate(model, test_loader)
        history["loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        history["test_acc"].append(epoch_test_acc)
        print(f"  epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}"
              f"  train_acc={epoch_acc:.4f}  test_acc={epoch_test_acc:.4f}")
    return history


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

The paper trains with plain SGD, momentum 0.9, weight decay 5e-4, and a hand-tuned
schedule built for 1.2 million images. With only a few hundred images here, that
schedule has no time to work, so we use `Adam` instead — it adapts its own step size
without manual tuning. (See
[adaptive_learning_rates.ipynb](adaptive_learning_rates.ipynb) for why.)

In [ ]:
print("Training AlexNet on the small CIFAR-10 subset...")
history = train_model(alexnet, train_loader, epochs=EPOCHS)

final_train_acc = history["train_acc"][-1]
final_test_acc = history["test_acc"][-1]
print(f"\nfinal train accuracy: {final_train_acc:.4f}")
print(f"final test accuracy:  {final_test_acc:.4f}")
print(f"train - test gap:     {final_train_acc - final_test_acc:.4f}")

In [ ]:
epochs_range = np.arange(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_range, history["loss"], "o-")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training loss")
axes[0].set_title("Training loss")

axes[1].plot(epochs_range, history["train_acc"], "o-", label="train accuracy")
axes[1].plot(epochs_range, history["test_acc"], "s-", label="test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title(f"Train vs. test accuracy\n({N_TRAIN_PER_CLASS * 10} train / {N_TEST_PER_CLASS * 10} test images)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
alexnet.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = alexnet(images.to(device)).argmax(1).cpu()

unnorm_mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
unnorm_std = torch.tensor(CIFAR_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = images[i] * unnorm_std + unnorm_mean
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    correct = preds[i] == labels[i]
    ax.set_title(f"pred: {class_names[preds[i]]}\ntrue: {class_names[labels[i]]}",
                 fontsize=8, color="green" if correct else "red")
    ax.axis("off")
fig.suptitle("AlexNet predictions on held-out test images (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

### 6. Summary

- AlexNet's ILSVRC-2012 win (top-5 error 15.3% vs. 26% for the best non-neural entry)
  is the result most often credited with starting the modern deep learning era in
  computer vision.
- Its core ideas — ReLU, overlapping max pooling, dropout, data augmentation — are
  still standard practice; LRN is the one piece later replaced, mostly by Batch
  Normalization.
- Section 3 showed AlexNet's ~60M parameters are concentrated almost entirely in the
  fully connected classifier (`fc6`, `fc7`), not the convolutional feature extractor
  — exactly why the paper leans on dropout and 1.2M training images to avoid
  overfitting.
- Section 5 makes that concrete at a tiny scale: run the training cell above and
  watch train accuracy climb while test accuracy lags well behind — the same
  architecture, overfitting on 300 images the way the paper's design choices exist
  to prevent.